# Hospital datasets cleaning and validation

In [4]:
import pandas as pd
from pathlib import Path

raw_path = Path("../data/raw")
processed_path = Path("../data/processed")

processed_path.mkdir(parents=True, exist_ok=True)

hospital = pd.read_csv(raw_path / "hospital_overview_dataset.csv")
flow = pd.read_csv(raw_path / "patient_flow_dataset.csv")
department = pd.read_csv(raw_path / "department_analytics_dataset.csv")
resource = pd.read_csv(raw_path / "resource_utilization_dataset.csv")

datasets = {
    "Hospital Overview": hospital,
    "Patient Flow": flow,
    "Department Analytics": department,
    "Resource Utilization": resource
}

In [5]:
for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * 50)
    print("Shape:", df.shape)
    print("Duplicate rows:", df.duplicated().sum())
    print("Missing values:", df.isna().sum().sum())

    missing = df.isna().sum()
    if missing.sum() > 0:
        print("\nColumns with missing values:")
        print(missing[missing > 0])


Hospital Overview
--------------------------------------------------
Shape: (465, 24)
Duplicate rows: 5
Missing values: 31

Columns with missing values:
insurance_type                 8
patient_satisfaction_score    23
dtype: int64

Patient Flow
--------------------------------------------------
Shape: (1134, 20)
Duplicate rows: 8
Missing values: 1402

Columns with missing values:
from_department_id              464
from_department_name            464
bed_id                          464
duration_in_department_hours     10
dtype: int64

Department Analytics
--------------------------------------------------
Shape: (2160, 24)
Duplicate rows: 6
Missing values: 1372

Columns with missing values:
avg_satisfaction_score    1372
dtype: int64

Resource Utilization
--------------------------------------------------
Shape: (6472, 24)
Duplicate rows: 10
Missing values: 10972

Columns with missing values:
nurses_on_duty_snapshot      2157
doctors_on_duty_snapshot     2157
avg_response_time_minute

In [6]:
# Remove duplicate rows
hospital = hospital.drop_duplicates().copy()
flow = flow.drop_duplicates().copy()
department = department.drop_duplicates().copy()
resource = resource.drop_duplicates().copy()


# Clean text fields
for df in [hospital, flow, department, resource]:
    text_cols = df.select_dtypes(include="object").columns
    for col in text_cols:
        df[col] = df[col].str.strip()


# Standardize common categorical values
hospital["patient_gender"] = hospital["patient_gender"].replace({
    "M": "Male",
    "m": "Male",
    "MALE": "Male",
    "F": "Female",
    "f": "Female",
    "FEMALE": "Female"
})

hospital["admission_type"] = hospital["admission_type"].replace({
    "ER": "Emergency",
    "EMERGENCY": "Emergency",
    "Emergency Admission": "Emergency",
    "URGENT": "Urgent",
    "URGENT ": "Urgent",
    "ELECTIVE": "Elective",
    "Elective Admission": "Elective"
})


# Standardize IDs
id_columns = {
    "admission_id": hospital,
    "patient_id": hospital,
    "hospital_id": hospital,
    "department_id": hospital,
    "movement_id": flow,
    "admission_id": flow,
    "patient_id": flow,
    "hospital_id": flow,
    "current_department_id": flow,
    "from_department_id": flow,
    "hospital_id": department,
    "department_id": department,
    "resource_utilization_id": resource,
    "hospital_id": resource,
    "department_id": resource
}

for column, df in id_columns.items():
    if column in df.columns:
        df[column] = df[column].astype("string").str.strip().str.upper()


# Convert date fields
for col in ["admission_date", "discharge_date"]:
    hospital[col] = pd.to_datetime(hospital[col], errors="coerce")

flow["movement_datetime"] = pd.to_datetime(
    flow["movement_datetime"], errors="coerce"
)
flow["movement_date"] = pd.to_datetime(
    flow["movement_date"], errors="coerce"
)

department["date"] = pd.to_datetime(
    department["date"], errors="coerce"
)

resource["date"] = pd.to_datetime(
    resource["date"], errors="coerce"
)

resource["last_maintenance_date"] = pd.to_datetime(
    resource["last_maintenance_date"], errors="coerce"
)

In [7]:
datasets = {
    "Hospital Overview": hospital,
    "Patient Flow": flow,
    "Department Analytics": department,
    "Resource Utilization": resource
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * 50)
    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Duplicate rows:", df.duplicated().sum())
    print("Missing values:", df.isna().sum().sum())


# Numeric validation
print("\nInvalid numeric values:")
print("Invalid ages:", ((hospital["patient_age"] < 0) | (hospital["patient_age"] > 120)).sum())
print("Invalid LOS:", (hospital["length_of_stay_days"] < 0).sum())
print("Invalid billing:", (hospital["total_bill_amount"] < 0).sum())
print("Invalid occupancy:", ((department["bed_occupancy_rate_pct"] < 0) | (department["bed_occupancy_rate_pct"] > 100)).sum())
print("Invalid utilization:", ((resource["utilization_rate_pct"] < 0) | (resource["utilization_rate_pct"] > 100)).sum())


# Foreign-key validation
unmatched_admissions = flow[
    ~flow["admission_id"].isin(hospital["admission_id"])
]

unmatched_patients = flow[
    ~flow["patient_id"].isin(hospital["patient_id"])
]

print("\nRelationship validation:")
print("Unmatched admission IDs:", len(unmatched_admissions))
print("Unmatched patient IDs:", len(unmatched_patients))


Hospital Overview
--------------------------------------------------
Rows: 460
Columns: 24
Duplicate rows: 0
Missing values: 30

Patient Flow
--------------------------------------------------
Rows: 1126
Columns: 20
Duplicate rows: 0
Missing values: 1390

Department Analytics
--------------------------------------------------
Rows: 2154
Columns: 24
Duplicate rows: 0
Missing values: 1370

Resource Utilization
--------------------------------------------------
Rows: 6462
Columns: 24
Duplicate rows: 0
Missing values: 10953

Invalid numeric values:
Invalid ages: 0
Invalid LOS: 0
Invalid billing: 0
Invalid occupancy: 0
Invalid utilization: 0

Relationship validation:
Unmatched admission IDs: 0
Unmatched patient IDs: 0


In [8]:
hospital.to_csv(
    processed_path / "hospital_overview_dataset.csv",
    index=False
)

flow.to_csv(
    processed_path / "patient_flow_dataset.csv",
    index=False
)

department.to_csv(
    processed_path / "department_analytics_dataset.csv",
    index=False
)

resource.to_csv(
    processed_path / "resource_utilization_dataset.csv",
    index=False
)

print("Cleaned datasets saved successfully.")

Cleaned datasets saved successfully.
